In [12]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [13]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker,
# to demonstrate that the attacker can be detected.

# BB84 with Attacker Overview:
# The protocol is identical to the plain version, but Eve intercepts the
# quantum channel between Alice and Bob.
#
# Attack strategy (intercept-resend):
#   Eve measures each qubit in a randomly chosen basis, then re-encodes and
#   forwards a new qubit to Bob based on her measurement result.
#   Because Eve does not know Alice's basis, she guesses wrong ~50% of the time.
#   When she guesses wrong, she disturbs the qubit state, introducing errors
#   that Alice and Bob can detect during error checking.
#
# Expected QBER with intercept-resend attack: ~25%
# Detection threshold used here: 15%
#
# Standard basis (basis=0):  |0> encodes bit 0,  |1> encodes bit 1
# Diagonal basis  (basis=1):  |+> encodes bit 0,  |-> encodes bit 1
#
# All random choices are generated by measuring a qubit in the
# |+> = 1/sqrt(2)(|0>+|1>) state � NOT Python's random module.

In [14]:
# -----------------------------------------------------------------------------
# QUANTUM RANDOM NUMBER GENERATOR
# -----------------------------------------------------------------------------
# Applying a Hadamard gate to |0> produces 1/sqrt(2)(|0>+|1>).
# Measuring this superposition collapses it to 0 or 1 with equal probability,
# giving us a true quantum random bit.

backend = BasicSimulator()

def quantum_random_bit():
    """Return a single random bit (0 or 1) via quantum measurement."""
    qc = QuantumCircuit(1, 1)
    qc.h(0)           # Create superposition |+> = 1/sqrt(2)(|0>+|1>)
    qc.measure(0, 0)  # Collapse to 0 or 1 with equal probability
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])


def quantum_random_bits(n):
    """Return a list of n random bits generated via quantum measurement."""
    return [quantum_random_bit() for _ in range(n)]


# Sanity check
print("Sample random bits:", quantum_random_bits(10))

Sample random bits: [1, 0, 0, 1, 0, 0, 0, 0, 0, 1]


In [15]:
# -----------------------------------------------------------------------------
# QUBIT ENCODING AND MEASUREMENT HELPERS
# -----------------------------------------------------------------------------
# BB84 uses two bases:
#   Standard basis (basis=0):  |0> encodes bit 0,  |1> encodes bit 1
#   Diagonal basis (basis=1):  |+> encodes bit 0,  |-> encodes bit 1
#
# Encoding:
#   Standard: start in |0>; apply X gate if bit=1  -> gives |0> or |1>
#   Diagonal: start in |0>; apply X if bit=1; then apply H -> gives |+> or |->
#
# Measurement:
#   Standard: measure directly in the Z basis (distinguishes |0> from |1>)
#   Diagonal: apply H first to rotate back to Z basis, then measure
#             (H|+> = |0>, H|-> = |1>)

def encode_qubit(bit, basis):
    """
    Encode a classical bit into a qubit using the specified basis.

    Parameters
    ----------
    bit   : int  -- 0 or 1, the value to encode
    basis : int  -- 0 for standard, 1 for diagonal

    Returns
    -------
    QuantumCircuit -- a 1-qubit circuit representing the encoded qubit
    """
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)   # Flip |0> to |1>
    if basis == 1:
        qc.h(0)   # Rotate to diagonal basis: |0>->|+>, |1>|->
    return qc


def measure_qubit(qubit_circuit, basis):
    """
    Measure a qubit circuit in the specified basis.

    Parameters
    ----------
    qubit_circuit : QuantumCircuit -- the encoded qubit (1-qubit, no classical bits)
    basis         : int            -- 0 for standard, 1 for diagonal

    Returns
    -------
    int -- the measured bit (0 or 1)
    """
    from qiskit import ClassicalRegister
    qc = qubit_circuit.copy()
    qc.add_register(ClassicalRegister(1))
    if basis == 1:
        qc.h(0)   # Rotate back from diagonal basis before measuring
    qc.measure(0, 0)
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

In [16]:
# -----------------------------------------------------------------------------
# ALICE -- KEY PREPARATION
# -----------------------------------------------------------------------------
# Alice generates random bit values and random basis choices, then encodes
# each bit as a qubit ready to send over the quantum channel.

def alice_prepare(n):
    """
    Alice prepares n qubits for transmission.

    Parameters
    ----------
    n : int -- number of qubits to prepare

    Returns
    -------
    bits   : list[int]            -- Alice's random bit values
    bases  : list[int]            -- Alice's random basis choices (0=standard, 1=diagonal)
    qubits : list[QuantumCircuit] -- encoded qubits to send
    """
    bits  = quantum_random_bits(n)   # Random bit values
    bases = quantum_random_bits(n)   # Random basis choices
    qubits = [encode_qubit(b, basis) for b, basis in zip(bits, bases)]
    return bits, bases, qubits

In [17]:
# -----------------------------------------------------------------------------
# CIRCUIT VISUALIZATIONS
# -----------------------------------------------------------------------------
# Show the four possible qubit encodings Alice can produce, so it is clear
# exactly which gates are applied for each (bit, basis) combination.
#
#   Standard basis, bit=0 : |0>  -- no gates needed, qubit starts in |0>
#   Standard basis, bit=1 : |1>  -- X gate flips |0> to |1>
#   Diagonal basis, bit=0 : |+>  -- H gate rotates |0> to |+>
#   Diagonal basis, bit=1 : |->  -- X then H rotates |0> to |->

print('=== Qubit Encoding Circuits ===')
labels = [
    ('Standard basis, bit=0  -->  |0>', 0, 0),
    ('Standard basis, bit=1  -->  |1>', 1, 0),
    ('Diagonal basis, bit=0  -->  |+>', 0, 1),
    ('Diagonal basis, bit=1  -->  |->', 1, 1),
]
for desc, bit, basis in labels:
    qc = encode_qubit(bit, basis)
    # Add a barrier to visually separate encoding from measurement
    qc.barrier()
    print(f'\n{desc}')
    print(qc.draw(output='text'))

=== Qubit Encoding Circuits ===

Standard basis, bit=0  -->  |0>
    ░ 
q: ─░─
    ░ 

Standard basis, bit=1  -->  |1>
   ┌───┐ ░ 
q: ┤ X ├─░─
   └───┘ ░ 

Diagonal basis, bit=0  -->  |+>
   ┌───┐ ░ 
q: ┤ H ├─░─
   └───┘ ░ 

Diagonal basis, bit=1  -->  |->
   ┌───┐┌───┐ ░ 
q: ┤ X ├┤ H ├─░─
   └───┘└───┘ ░ 


In [18]:
# -----------------------------------------------------------------------------
# EVE -- ATTACK STRATEGIES
# -----------------------------------------------------------------------------
# Two attack strategies are implemented:
#
# 1. FULL INTERCEPT-RESEND
#    Eve intercepts EVERY qubit, measures it in a random basis, re-encodes
#    her result, and forwards the new qubit to Bob.
#    Expected QBER: ~25%  (reliably detected above the 15% threshold)
#
# 2. PARTIAL INTERCEPT
#    Eve only intercepts ~50% of qubits at random; the rest pass undisturbed.
#    Expected QBER: ~12.5%  (sits just below the 15% threshold, harder to detect)
#    This shows that a stealthy attacker can stay under the radar with a
#    lower intercept rate, at the cost of learning less of the key.

def eve_intercept(qubits):
    """
    Eve intercepts and re-sends EVERY qubit (full intercept-resend attack).

    Parameters
    ----------
    qubits : list[QuantumCircuit] -- qubits sent by Alice

    Returns
    -------
    intercepted_qubits : list[QuantumCircuit] -- re-encoded qubits forwarded to Bob
    eve_bases          : list[int]            -- Eve's random basis choices
    eve_bits           : list[int]            -- Eve's measurement results
    """
    eve_bases = quantum_random_bits(len(qubits))  # Random basis for every qubit
    eve_bits  = []
    intercepted_qubits = []

    for qubit, basis in zip(qubits, eve_bases):
        # Measure the qubit -- this collapses the original quantum state
        measured_bit = measure_qubit(qubit, basis)
        eve_bits.append(measured_bit)
        # Re-encode and forward to Bob
        intercepted_qubits.append(encode_qubit(measured_bit, basis))

    return intercepted_qubits, eve_bases, eve_bits


def eve_partial_intercept(qubits):
    """
    Eve intercepts ~50% of qubits at random (partial intercept attack).

    Each qubit is independently intercepted or passed through based on a
    quantum random bit (0 = intercept, 1 = pass through).
    Qubits that are not intercepted reach Bob completely undisturbed.

    Expected QBER: ~12.5% -- just below the 15% detection threshold.

    Parameters
    ----------
    qubits : list[QuantumCircuit] -- qubits sent by Alice

    Returns
    -------
    forwarded_qubits : list[QuantumCircuit] -- qubits forwarded to Bob
    eve_bases        : list[int|None]       -- Eve's basis per qubit (None = not intercepted)
    eve_bits         : list[int|None]       -- Eve's result per qubit (None = not intercepted)
    intercept_mask   : list[bool]           -- True where Eve intercepted
    """
    forwarded_qubits = []
    eve_bases        = []
    eve_bits         = []
    intercept_mask   = []

    for qubit in qubits:
        # Use a quantum random bit to decide whether to intercept this qubit
        do_intercept = (quantum_random_bit() == 0)  # 50% chance
        intercept_mask.append(do_intercept)

        if do_intercept:
            # Eve intercepts: measure in a random basis and re-encode
            basis = quantum_random_bit()
            measured_bit = measure_qubit(qubit, basis)
            eve_bases.append(basis)
            eve_bits.append(measured_bit)
            forwarded_qubits.append(encode_qubit(measured_bit, basis))
        else:
            # Eve lets this qubit pass through completely undisturbed
            eve_bases.append(None)
            eve_bits.append(None)
            forwarded_qubits.append(qubit)

    n_intercepted = sum(intercept_mask)
    print(f'[Eve] Partial intercept: {n_intercepted}/{len(qubits)} qubits intercepted '
          f'({n_intercepted/len(qubits):.0%})')
    return forwarded_qubits, eve_bases, eve_bits, intercept_mask

In [19]:
# -----------------------------------------------------------------------------
# BOB -- MEASUREMENT
# -----------------------------------------------------------------------------
# Bob receives the (possibly tampered) qubits from the channel and measures
# each one in a randomly chosen basis.

def bob_measure(qubits):
    """
    Bob measures each received qubit in a randomly chosen basis.

    Parameters
    ----------
    qubits : list[QuantumCircuit] -- qubits received from the channel

    Returns
    -------
    measured_bits : list[int] -- Bob's measurement outcomes
    bases         : list[int] -- Bob's random basis choices
    """
    bases = quantum_random_bits(len(qubits))   # Bob's random basis choices
    measured_bits = [measure_qubit(q, b) for q, b in zip(qubits, bases)]
    return measured_bits, bases

In [20]:
# -----------------------------------------------------------------------------
# SIFTING -- BASIS RECONCILIATION
# -----------------------------------------------------------------------------
# Alice and Bob publicly compare their basis choices over a classical channel.
# They keep only the positions where both chose the same basis.
# The corresponding bits form the sifted key.

def sift_key(alice_bits, alice_bases, bob_bits, bob_bases):
    """
    Retain only the bits where Alice and Bob used the same basis.

    Parameters
    ----------
    alice_bits  : list[int] -- Alice's original bit values
    alice_bases : list[int] -- Alice's basis choices
    bob_bits    : list[int] -- Bob's measured bit values
    bob_bases   : list[int] -- Bob's basis choices

    Returns
    -------
    alice_key : list[int] -- Alice's sifted key
    bob_key   : list[int] -- Bob's sifted key
    """
    alice_key, bob_key = [], []
    for i in range(len(alice_bits)):
        if alice_bases[i] == bob_bases[i]:   # Bases match -- keep this bit
            alice_key.append(alice_bits[i])
            bob_key.append(bob_bits[i])
    return alice_key, bob_key

In [21]:
# -----------------------------------------------------------------------------
# ERROR CHECKING
# -----------------------------------------------------------------------------
# Alice and Bob sacrifice a portion of their sifted key to estimate the
# Quantum Bit Error Rate (QBER).  A QBER above the threshold signals an attack.
#
# With an intercept-resend attack the expected QBER is ~25%, which is well
# above the 15% threshold used here.

DETECTION_THRESHOLD = 0.15   # Abort if more than 15% of check bits are wrong

def check_errors(alice_key, bob_key, sample_fraction=0.5):
    """
    Estimate the QBER by comparing a sample of the sifted key.

    Parameters
    ----------
    alice_key       : list[int] -- Alice's sifted key
    bob_key         : list[int] -- Bob's sifted key
    sample_fraction : float     -- fraction of the key used for error checking

    Returns
    -------
    error_rate      : float -- proportion of mismatched bits in the sample
    attack_detected : bool  -- True if error_rate exceeds DETECTION_THRESHOLD
    """
    n_sample = max(1, int(len(alice_key) * sample_fraction))
    errors = sum(a != b for a, b in zip(alice_key[:n_sample], bob_key[:n_sample]))
    error_rate = errors / n_sample
    attack_detected = error_rate > DETECTION_THRESHOLD
    return error_rate, attack_detected

In [22]:
# -----------------------------------------------------------------------------
# PROTOCOL TABLE DISPLAY
# -----------------------------------------------------------------------------
# Prints a formatted table matching the lecture slide layout.
# Used twice in the attacker scenario:
#   1. Alice -> Eve  (what Eve sees on the quantum channel)
#   2. Alice -> Bob  (what Bob receives after Eve's interference)
#
# Qubit symbols: standard basis -> 0 / 1 ; diagonal basis -> + / -
# Columns marked * are positions where both parties' bases matched.
# The receiver's bit shows ? where bases did not match.

def qubit_symbol(bit, basis):
    """Return the qubit state symbol for a given bit and basis."""
    if basis == 0:          # Standard basis: |0> or |1>
        return str(bit)
    else:                   # Diagonal basis: |+> or |->
        return '+' if bit == 0 else '-'


def print_bb84_table(title, alice_bits, alice_bases,
                     receiver_bits, receiver_bases, receiver_label='B'):
    """
    Print a BB84 protocol table to the console.

    Parameters
    ----------
    title          : str       -- heading printed above the table
    alice_bits     : list[int] -- Alice's raw bit values
    alice_bases    : list[int] -- Alice's basis choices (0=standard, 1=diagonal)
    receiver_bits  : list[int] -- receiver's measured bit values
    receiver_bases : list[int] -- receiver's basis choices
    receiver_label : str       -- single-character label for the receiver row (default 'B')
    """
    n   = len(alice_bits)
    col = 3   # fixed column width

    # Positions where Alice's and receiver's bases match
    match = [alice_bases[i] == receiver_bases[i] for i in range(n)]

    idx_row   = [str(i)                                         for i in range(n)]
    match_row = ['*' if match[i] else ' '                       for i in range(n)]
    a_bit_row = [str(alice_bits[i])                             for i in range(n)]
    a_bas_row = ['s' if alice_bases[i] == 0 else 'd'            for i in range(n)]
    qubit_row = [qubit_symbol(alice_bits[i], alice_bases[i])    for i in range(n)]
    r_bas_row = ['s' if receiver_bases[i] == 0 else 'd'         for i in range(n)]
    r_bit_row = [str(receiver_bits[i]) if match[i] else '?'     for i in range(n)]

    def fmt_row(label, cells):
        return f'{label:<10}' + ''.join(c.center(col) for c in cells)

    sep = '-' * (10 + col * n)
    print(f'\n{title}')
    print(sep)
    print(fmt_row('Index',              idx_row))
    print(fmt_row('Match',              match_row))   # * = bases matched
    print(sep)
    print(fmt_row('A bit',              a_bit_row))
    print(fmt_row('A basis',            a_bas_row))   # s=standard, d=diagonal
    print(fmt_row('Qubit',              qubit_row))   # 0/1 or +/-
    print(sep)
    print(fmt_row(f'{receiver_label} basis', r_bas_row))
    print(fmt_row(f'{receiver_label} bit',   r_bit_row))  # ? where bases differ
    print(sep)


# -----------------------------------------------------------------------------
# FULL BB84 PROTOCOL -- WITH ATTACKER (INTERCEPT-RESEND)
# -----------------------------------------------------------------------------
# Runs the complete BB84 protocol with Eve performing an intercept-resend attack.

def run_bb84_attacker(n_qubits=20):
    """
    Simulate the BB84 protocol with an intercept-resend attacker (Eve).

    Parameters
    ----------
    n_qubits : int -- number of qubits Alice sends

    Returns
    -------
    final_key : list[int] -- shared key if no attack detected, else empty list
    """
    print(f'=== BB84 Protocol (Intercept-Resend Attack) -- {n_qubits} qubits ===')

    # -- ALICE ----------------------------------------------------------------
    alice_bits, alice_bases, qubits = alice_prepare(n_qubits)

    # -- EVE (quantum channel) ------------------------------------------------
    # Eve intercepts every qubit, measures it in a random basis, and re-sends.
    intercepted, eve_bases, eve_bits = eve_intercept(qubits)

    # Table 1: Alice vs Eve -- shows what Eve intercepted
    # * marks positions where Eve guessed Alice's basis correctly (no disturbance)
    print_bb84_table(
        'TABLE 1 -- Alice -> Eve (what Eve intercepted)',
        alice_bits, alice_bases,
        eve_bits,   eve_bases,
        receiver_label='E'
    )

    # -- BOB ------------------------------------------------------------------
    # Bob receives Eve's re-sent qubits (not Alice's originals)
    bob_bits, bob_bases = bob_measure(intercepted)

    # Table 2: Alice vs Bob -- shows the errors Eve introduced
    # * marks positions where Alice and Bob's bases matched (sifted positions)
    # Errors in the sifted key are visible where A bit != B bit on * columns
    print_bb84_table(
        'TABLE 2 -- Alice -> Bob (after Eve\'s interference)',
        alice_bits, alice_bases,
        bob_bits,   bob_bases,
        receiver_label='B'
    )

    # -- SIFTING (public classical channel) -----------------------------------
    alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    print(f'\n[Sifting] Alice sifted key : {alice_key}')
    print(f'[Sifting] Bob   sifted key : {bob_key}')

    # -- ERROR CHECKING -------------------------------------------------------
    error_rate, attack_detected = check_errors(alice_key, bob_key)
    print(f'\n[Error Check] QBER = {error_rate:.2%}  (threshold = {DETECTION_THRESHOLD:.0%})')

    if attack_detected:
        print('[Error Check] Attack DETECTED -- aborting protocol.')
        return []
    else:
        # Unlikely with a full intercept-resend attack, but possible with
        # small qubit counts due to statistical fluctuation.
        print('[Error Check] No attack detected (channel appears secure).')

    # -- FINAL KEY ------------------------------------------------------------
    n_check = max(1, int(len(alice_key) * 0.5))
    final_key = alice_key[n_check:]
    print(f'\n[Key] Final shared key ({len(final_key)} bits): {final_key}')
    return final_key


# Run the protocol with the attacker
key = run_bb84_attacker(n_qubits=20)

=== BB84 Protocol (Intercept-Resend Attack) -- 20 qubits ===

TABLE 1 -- Alice -> Eve (what Eve intercepted)
----------------------------------------------------------------------
Index      0  1  2  3  4  5  6  7  8  9  10 11 12 13 14 15 16 17 18 19
Match      *  *  *     *  *     *                                *    
----------------------------------------------------------------------
A bit      0  1  1  1  1  0  0  1  0  0  0  0  0  0  0  0  0  0  1  0 
A basis    s  d  d  s  d  s  d  d  d  s  d  s  d  s  s  d  d  d  s  s 
Qubit      0  -  -  1  -  0  +  -  +  0  +  0  +  0  0  +  +  +  1  0 
----------------------------------------------------------------------
E basis    s  d  d  d  d  s  s  d  s  d  s  d  s  d  d  s  s  s  s  d 
E bit      0  1  1  ?  1  0  ?  1  ?  ?  ?  ?  ?  ?  ?  ?  ?  ?  1  ? 
----------------------------------------------------------------------

TABLE 2 -- Alice -> Bob (after Eve's interference)
----------------------------------------------------------

In [23]:
# -----------------------------------------------------------------------------
# FULL BB84 PROTOCOL -- WITH PARTIAL INTERCEPT ATTACK
# -----------------------------------------------------------------------------
# Eve only intercepts ~50% of qubits.  The expected QBER is ~12.5%, which
# sits just below the 15% detection threshold -- demonstrating that a
# stealthy attacker can sometimes evade detection at the cost of learning
# only a fraction of the key.

def run_bb84_partial(n_qubits=20):
    """
    Simulate the BB84 protocol with a partial intercept attack (Eve ~50%).

    Parameters
    ----------
    n_qubits : int -- number of qubits Alice sends

    Returns
    -------
    final_key : list[int] -- shared key if no attack detected, else empty list
    """
    print(f'=== BB84 Protocol (Partial Intercept Attack) -- {n_qubits} qubits ===')

    # -- ALICE ----------------------------------------------------------------
    alice_bits, alice_bases, qubits = alice_prepare(n_qubits)

    # -- EVE (quantum channel) ------------------------------------------------
    # Eve intercepts ~50% of qubits; the rest pass through undisturbed.
    forwarded, eve_bases, eve_bits, mask = eve_partial_intercept(qubits)

    # Table 1: Alice vs Eve -- only intercepted positions are shown with Eve's result
    # Non-intercepted positions show None in eve_bases; we substitute 's' for display
    eve_bases_display = [b if b is not None else alice_bases[i]
                         for i, b in enumerate(eve_bases)]
    eve_bits_display  = [b if b is not None else alice_bits[i]
                         for i, b in enumerate(eve_bits)]
    print_bb84_table(
        'TABLE 1 -- Alice -> Eve (intercepted positions only; others pass through)',
        alice_bits, alice_bases,
        eve_bits_display, eve_bases_display,
        receiver_label='E'
    )

    # -- BOB ------------------------------------------------------------------
    bob_bits, bob_bases = bob_measure(forwarded)

    # Table 2: Alice vs Bob -- errors only appear at intercepted positions
    print_bb84_table(
        'TABLE 2 -- Alice -> Bob (errors only at intercepted positions)',
        alice_bits, alice_bases,
        bob_bits, bob_bases,
        receiver_label='B'
    )

    # -- SIFTING --------------------------------------------------------------
    alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    print(f'\n[Sifting] Alice sifted key : {alice_key}')
    print(f'[Sifting] Bob   sifted key : {bob_key}')

    # -- ERROR CHECKING -------------------------------------------------------
    error_rate, attack_detected = check_errors(alice_key, bob_key)
    print(f'\n[Error Check] QBER = {error_rate:.2%}  (threshold = {DETECTION_THRESHOLD:.0%})')

    if attack_detected:
        print('[Error Check] Attack DETECTED -- aborting protocol.')
        return []
    else:
        print('[Error Check] Attack NOT detected -- Eve stayed under the threshold.')

    # -- FINAL KEY ------------------------------------------------------------
    n_check = max(1, int(len(alice_key) * 0.5))
    final_key = alice_key[n_check:]
    print(f'\n[Key] Final shared key ({len(final_key)} bits): {final_key}')
    return final_key


# Run the partial intercept scenario
key_partial = run_bb84_partial(n_qubits=20)

=== BB84 Protocol (Partial Intercept Attack) -- 20 qubits ===
[Eve] Partial intercept: 9/20 qubits intercepted (45%)

TABLE 1 -- Alice -> Eve (intercepted positions only; others pass through)
----------------------------------------------------------------------
Index      0  1  2  3  4  5  6  7  8  9  10 11 12 13 14 15 16 17 18 19
Match      *     *  *  *     *  *     *  *  *  *  *  *  *  *  *  *  * 
----------------------------------------------------------------------
A bit      0  0  1  0  1  1  1  0  0  0  1  0  0  1  1  0  0  1  0  1 
A basis    d  s  d  d  s  s  s  s  s  s  d  s  d  s  d  d  s  s  d  s 
Qubit      +  0  -  +  1  1  1  0  0  0  -  0  +  1  -  +  0  1  +  1 
----------------------------------------------------------------------
E basis    d  d  d  d  s  d  s  s  d  s  d  s  d  s  d  d  s  s  d  s 
E bit      0  ?  1  0  1  ?  1  0  ?  0  1  0  0  1  1  0  0  1  0  1 
----------------------------------------------------------------------

TABLE 2 -- Alice -> Bob (e

In [24]:
# -----------------------------------------------------------------------------
# VERIFICATION -- COMPARING BOTH ATTACK TYPES OVER MULTIPLE TRIALS
# -----------------------------------------------------------------------------
# Run 10 trials of each attack and compare detection rates and average QBERs.
#
# Full intercept:    QBER ~25% -- should be detected in nearly every trial
# Partial intercept: QBER ~12.5% -- may slip under the 15% threshold

N_TRIALS  = 10
N_QUBITS  = 30

print(f'Running {N_TRIALS} trials of each attack ({N_QUBITS} qubits per trial)...\n')

# -- Full intercept-resend ---------------------------------------------------
print('--- Full Intercept-Resend ---')
full_detected = 0
full_qbers    = []
for trial in range(1, N_TRIALS + 1):
    a_bits, a_bases, qubits = alice_prepare(N_QUBITS)
    intercepted, _, _       = eve_intercept(qubits)
    b_bits, b_bases         = bob_measure(intercepted)
    a_key, b_key            = sift_key(a_bits, a_bases, b_bits, b_bases)
    rate, detected          = check_errors(a_key, b_key)
    full_detected += int(detected)
    full_qbers.append(rate)
    status = 'DETECTED' if detected else 'missed'
    print(f'  Trial {trial:2d}: sifted={len(a_key):2d}  QBER={rate:.2%}  --> {status}')

avg_full = sum(full_qbers) / len(full_qbers)
print(f'\n  Detected: {full_detected}/{N_TRIALS}  |  Avg QBER: {avg_full:.2%}')

# -- Partial intercept (~50%) ------------------------------------------------
print('\n--- Partial Intercept (~50%) ---')
partial_detected = 0
partial_qbers    = []
for trial in range(1, N_TRIALS + 1):
    a_bits, a_bases, qubits          = alice_prepare(N_QUBITS)
    forwarded, _, _, _               = eve_partial_intercept(qubits)
    b_bits, b_bases                  = bob_measure(forwarded)
    a_key, b_key                     = sift_key(a_bits, a_bases, b_bits, b_bases)
    rate, detected                   = check_errors(a_key, b_key)
    partial_detected += int(detected)
    partial_qbers.append(rate)
    status = 'DETECTED' if detected else 'missed'
    print(f'  Trial {trial:2d}: sifted={len(a_key):2d}  QBER={rate:.2%}  --> {status}')

avg_partial = sum(partial_qbers) / len(partial_qbers)
print(f'\n  Detected: {partial_detected}/{N_TRIALS}  |  Avg QBER: {avg_partial:.2%}')

# -- Summary -----------------------------------------------------------------
print('\n=== Summary ===')
print(f'  Full intercept    -- detected {full_detected}/{N_TRIALS} trials, avg QBER {avg_full:.2%}')
print(f'  Partial intercept -- detected {partial_detected}/{N_TRIALS} trials, avg QBER {avg_partial:.2%}')
print('\nFull intercept should be detected reliably (~25% QBER).')
print('Partial intercept may evade detection (~12.5% QBER, threshold 15%).')

Running 10 trials of each attack (30 qubits per trial)...

--- Full Intercept-Resend ---
  Trial  1: sifted=11  QBER=40.00%  --> DETECTED
  Trial  2: sifted=18  QBER=22.22%  --> DETECTED
  Trial  3: sifted=15  QBER=14.29%  --> missed
  Trial  4: sifted=16  QBER=25.00%  --> DETECTED
  Trial  5: sifted=17  QBER=25.00%  --> DETECTED
  Trial  6: sifted=16  QBER=12.50%  --> missed
  Trial  7: sifted= 7  QBER=0.00%  --> missed
  Trial  8: sifted=15  QBER=57.14%  --> DETECTED
  Trial  9: sifted=18  QBER=22.22%  --> DETECTED
  Trial 10: sifted=15  QBER=14.29%  --> missed

  Detected: 6/10  |  Avg QBER: 23.27%

--- Partial Intercept (~50%) ---
[Eve] Partial intercept: 11/30 qubits intercepted (37%)
  Trial  1: sifted= 7  QBER=0.00%  --> missed
[Eve] Partial intercept: 19/30 qubits intercepted (63%)
  Trial  2: sifted=14  QBER=0.00%  --> missed
[Eve] Partial intercept: 18/30 qubits intercepted (60%)
  Trial  3: sifted=12  QBER=0.00%  --> missed
[Eve] Partial intercept: 14/30 qubits intercepted (